
# 09 — Learning a Custom Codebook Function *g* for Misdirection Analysis

---

## Overview and Theoretical Motivation

This notebook implements the next phase of our cryptic crossword misdirection project:
learning a **custom codebook function** $g$ by fine-tuning CALE via triplet training, then
using the learned $g$ to estimate the Average Treatment Effect (ATE) of misdirection.

---

## §0 — Imports and Configuration

Standard setup following the project convention established in notebooks 02–08.
We add PyTorch and HuggingFace transformers for model fine-tuning.


In [ ]:
import os
import re
import sys
import time
import warnings
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from collections import defaultdict, Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

import nltk
from nltk.corpus import wordnet as wn

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
import importlib.metadata, sys, subprocess, re

# Regular packages — installed from PyPI
REQUIRED = {
    "transformers":  "5.0.0",
    "tokenizers":    "0.22.2",
    "datasets":      "4.0.0",
    "accelerate":    "1.13.0",
    "numpy":         "2.0.2",
    "pandas":        "2.2.2",
    "pyarrow":       "18.1.0",
    "scikit-learn":  "1.6.1",
    "matplotlib":    "3.10.0",
    "seaborn":       "0.13.2",
    "tqdm":          "4.67.3",
    "nltk":          "3.9.1",
}

# Torch-family packages — must come from the PyTorch wheel index
# Store without +cu suffix; parse_version handles the comparison correctly
TORCH_REQUIRED = {
    "torch":         "2.10.0",
    "torchvision":   "0.25.0",
}

TORCH_INDEX = "https://download.pytorch.org/whl/cu128"

def parse_version(v):
    parts = []
    for segment in re.split(r"[.+]", v):
        if segment.isdigit():
            parts.append(int(segment))
        else:
            break
    return tuple(parts)

def check_packages(packages):
    to_install = []
    for pkg, exact_ver in packages.items():
        try:
            installed = importlib.metadata.version(pkg)
            ok = parse_version(installed) == parse_version(exact_ver)
            status = "✓" if ok else "✗  will reinstall"
            if not ok:
                to_install.append(f"{pkg}=={exact_ver}")
            print(f"  {pkg:<20} {installed:<20} (need == {exact_ver})  {status}")
        except importlib.metadata.PackageNotFoundError:
            print(f"  {pkg:<20} NOT INSTALLED        ✗  will install")
            to_install.append(f"{pkg}=={exact_ver}")
    return to_install

print(f"Python: {sys.version.split()[0]}\n")

print("── Standard packages ──────────────────────────────────────")
pypi_installs = check_packages(REQUIRED)

print("\n── Torch packages ─────────────────────────────────────────")
torch_installs = check_packages(TORCH_REQUIRED)

if pypi_installs:
    print(f"\nInstalling from PyPI: {', '.join(pypi_installs)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pypi_installs])

if torch_installs:
    print(f"\nInstalling from PyTorch index: {', '.join(torch_installs)}")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--index-url", TORCH_INDEX,
        *torch_installs
    ])

if pypi_installs or torch_installs:
    print("\n✓ Done. Restart the kernel if torch or torchvision were reinstalled.")
else:
    print("\n✓ All packages match Colab versions exactly — nothing to install.")

In [ ]:
# --- Environment Auto-Detection ---
# Same pattern as notebooks 02–08: detect Colab, Great Lakes, or local
# and set paths accordingly.
try:
    IS_COLAB = 'google.colab' in str(get_ipython())
except NameError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SIADS 692 Milestone II/'
                        'Milestone II - NLP Cryptic Crossword Clues')
else:
    try:
        PROJECT_ROOT = Path(__file__).resolve().parent.parent
    except NameError:
        PROJECT_ROOT = Path.cwd().parent

DATA_DIR        = PROJECT_ROOT / 'data'
OUTPUT_DIR      = PROJECT_ROOT / 'outputs'
SCRIPTS_DIR     = PROJECT_ROOT / 'scripts'
MODELS_DIR      = PROJECT_ROOT / 'models'

# Path to the pre-built harder dataset from NB 05.
# Check data/ first, then the notebook directory.
HARDER_PARQUET = DATA_DIR / 'dataset_harder.parquet'
if not HARDER_PARQUET.exists():
    HARDER_PARQUET = Path.cwd() / 'dataset_harder.parquet'
    if not HARDER_PARQUET.exists():
        raise FileNotFoundError(
            'dataset_harder.parquet not found in data/ or the notebook directory. '
            'Run NB 05 first, or place the file in one of these locations.'
        )

for d in [OUTPUT_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Add scripts/ to sys.path so feature_utils is importable ---
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Device selection: prefer CUDA, fall back to MPS (Apple Silicon), then CPU.
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

# Batch sizes — adjust for your GPU VRAM.
EMBED_BATCH_SIZE = 32 if IS_COLAB else 64
TRAIN_BATCH_SIZE = 16 if IS_COLAB else 32

# --- Sample mode ---
# When True, subsample data for fast iteration (development/debugging).
# Set to False for final runs.
SAMPLE_MODE = True
SAMPLE_SIZE = 20_000  # rows per class when sampling

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print(f'Environment: {"Google Colab" if IS_COLAB else "Local / Great Lakes"}')
print(f'Device:      {DEVICE}')
print(f'Sample mode: {SAMPLE_MODE}')
print(f'Project:     {PROJECT_ROOT}')
print(f'Harder dataset: {HARDER_PARQUET}')


---

# Phase 1: Triplet Training Data Construction from `dataset_harder.parquet`

## §1 — Load Dataset and Train/Test Split

The harder dataset from NB 05 already contains 240,211 real definition–answer pairs
and 240,211 matched distractor pairs (cosine-similarity-based, top-100 candidates).
Each distractor shares the same `clue_id` and `definition_wn` as its real pair, so we
can directly extract (anchor, positive, negative) triplets by pairing label=1 rows
with their corresponding label=0 rows.

We split at the (definition, answer) pair level to prevent leakage, consistent with
M2's GroupKFold approach (Decision 7).


In [ ]:
# ============================================================
# Load the harder dataset from NB 05
# ============================================================
# This parquet file is the ONLY required input.
# It contains 480,422 rows:
#   - 240,211 real (definition, answer) pairs (label=1)
#   - 240,211 distractor pairs (label=0), matched by clue_id
#
# Key column notes:
#   - surface_normalized: clue surface text (no CALE delimiters)
#   - definition / definition_wn: the definition substring (original case / lowercase)
#   - answer: the real answer in ALL CAPS (same for both label=1 and label=0)
#   - answer_wn: for label=1 = real answer (lowercase);
#                for label=0 = DISTRACTOR word (lowercase)
#   - distractor_source: for label=0, the distractor word (matches answer_wn)

df_harder = pd.read_parquet(HARDER_PARQUET)
print(f'Loaded dataset_harder.parquet: {len(df_harder):,} rows, {len(df_harder.columns)} columns')
print(f'Label distribution:')
print(df_harder['label'].value_counts().to_string())

# Separate real pairs (label=1) and distractor pairs (label=0)
df_real = df_harder[df_harder['label'] == 1].copy()
df_dist = df_harder[df_harder['label'] == 0].copy()
print(f'\nReal pairs:      {len(df_real):,}')
print(f'Distractor pairs: {len(df_dist):,}')

# Build the pair key used throughout the project
df_real['pair_key'] = df_real['definition_wn'] + '|||' + df_real['answer_wn']


In [ ]:
# ============================================================
# Train/Test Split at the (definition, answer) pair level
# ============================================================
# Split the REAL pairs (label=1) into train and test at the
# (definition_wn, answer_wn) pair level — same grouping as M2's
# GroupKFold CV (Decision 7).

unique_pairs = df_real['pair_key'].unique()
n_pairs = len(unique_pairs)
print(f'Unique (definition, answer) pairs: {n_pairs:,}')

# Deterministic 80/20 split at the pair level.
rng = np.random.RandomState(RANDOM_SEED)
pair_indices = rng.permutation(n_pairs)
split_point = int(0.8 * n_pairs)

train_pairs = set(unique_pairs[pair_indices[:split_point]])
test_pairs  = set(unique_pairs[pair_indices[split_point:]])

df_real['split'] = df_real['pair_key'].apply(
    lambda p: 'train' if p in train_pairs else 'test'
)

df_train_real = df_real[df_real['split'] == 'train'].copy()
df_test_real  = df_real[df_real['split'] == 'test'].copy()

print(f'Train set: {len(df_train_real):,} real rows  ({len(train_pairs):,} unique pairs)')
print(f'Test set:  {len(df_test_real):,} real rows   ({len(test_pairs):,} unique pairs)')
print(f'Train %:   {len(df_train_real)/len(df_real)*100:.1f}%')


## §2 — Constructing CALE Phrases and Extracting Triplets

Since `dataset_harder.parquet` is our only input, we construct the CALE-delimited
text phrases on-the-fly from two sources:

| Phrase type | Source columns | Construction |
|-------------|---------------|--------------|
| **Anchor** (contextualized definition) | `surface_normalized` + `definition` | Find `definition` within `surface_normalized` and wrap with `<t></t>` delimiters |
| **Positive / Negative** (decontextualized word) | `answer_wn` (label=1 or label=0) | `"<t>word</t>: {WordNet synset definition}"` — or bare `"<t>word</t>"` if no synset exists |

For each real pair (label=1), the corresponding distractor (label=0) shares the
same `clue_id`. The distractor word is in the label=0 row's `answer_wn` column.


In [ ]:
# ============================================================
# §2.1 — Build CALE Anchor Phrases (Contextualized Definitions)
# ============================================================
# The anchor text is the clue surface with <t></t> delimiters around the
# definition span — this is what CALE uses to produce a context-sensitive
# embedding of the definition.
#
# The definition appears as a contiguous substring of surface_normalized
# (guaranteed by NB 01's definition-in-surface and definition-at-edge
# filters). We perform a case-insensitive search using word boundaries
# to place the delimiters accurately.

def build_cale_anchor(surface, definition):
    '''Wrap the definition span within the clue surface with <t></t> delimiters.

    Uses case-insensitive word-boundary matching to locate the definition
    substring. Returns the delimited string, or None if the definition
    cannot be found.

    Example:
        surface = "theatre music"
        definition = "Theatre"
        returns "\x3ct>theatre\x3c/t> music"
    '''
    # Escape regex special characters in the definition
    pattern = re.escape(definition)
    # Case-insensitive search with word boundaries
    match = re.search(r'\b' + pattern + r'\b', surface, re.IGNORECASE)
    if match:
        start, end = match.start(), match.end()
        return surface[:start] + '<t>' + surface[start:end] + '</t>' + surface[end:]

    # Fallback: try without word boundaries (handles edge cases like
    # definitions at string boundaries or with hyphens)
    match = re.search(pattern, surface, re.IGNORECASE)
    if match:
        start, end = match.start(), match.end()
        return surface[:start] + '<t>' + surface[start:end] + '</t>' + surface[end:]

    return None


# Build anchor phrases for all real rows
df_train_real['cale_phrase'] = df_train_real.apply(
    lambda row: build_cale_anchor(row['surface_normalized'], row['definition']),
    axis=1
)
df_test_real['cale_phrase'] = df_test_real.apply(
    lambda row: build_cale_anchor(row['surface_normalized'], row['definition']),
    axis=1
)

# Report success rate
train_ok = df_train_real['cale_phrase'].notna().sum()
test_ok  = df_test_real['cale_phrase'].notna().sum()
print(f'CALE anchor phrases built:')
print(f'  Train: {train_ok:,} / {len(df_train_real):,} '
      f'({train_ok/len(df_train_real)*100:.1f}% success)')
print(f'  Test:  {test_ok:,} / {len(df_test_real):,} '
      f'({test_ok/len(df_test_real)*100:.1f}% success)')

# Drop rows where we couldn't place delimiters
df_train_real = df_train_real.dropna(subset=['cale_phrase']).copy()
df_test_real  = df_test_real.dropna(subset=['cale_phrase']).copy()

# Show a few examples
print(f'\nExample anchor phrases:')
for _, row in df_train_real.head(5).iterrows():
    print(f'  "{row["cale_phrase"]}"')


In [ ]:
# ============================================================
# §2.2 — Build Decontextualized CALE Phrases from WordNet
# ============================================================
# For positive (real answer) and negative (distractor) components,
# CALE expects a neutral context sentence with <t></t> around the
# target word. We construct these using WordNet synset definitions:
#
#   "<t>word</t>: synset_definition"
#
# e.g., "<t>fleet</t>: a group of ships or vehicles"
#
# If no WordNet synset exists, we fall back to just "<t>word</t>".

def build_decontext_phrase(word):
    '''Build a decontextualized CALE phrase for a single word.

    Uses the first (most common) WordNet synset definition as neutral context.
    '''
    wn_word = word.replace(' ', '_')
    synsets = wn.synsets(wn_word)
    if synsets:
        definition = synsets[0].definition()
        return f'<t>{word}</t>: {definition}'
    return f'<t>{word}</t>'


# Collect all unique words that need phrases: real answers + distractors
# + definition words (for Phase 3 decontextualized embeddings)
all_answer_words = set(df_real['answer_wn'].unique())
all_distractor_words = set(df_dist['answer_wn'].unique())
all_def_words = set(df_real['definition_wn'].unique())
all_words = all_answer_words | all_distractor_words | all_def_words

print(f'Building decontextualized CALE phrases for {len(all_words):,} unique words...')

# Build lookup: word -> CALE phrase
word_phrase_lookup = {}
n_with_synset = 0
for word in tqdm(sorted(all_words), desc='WordNet phrases'):
    phrase = build_decontext_phrase(word)
    word_phrase_lookup[word] = phrase
    if ':' in phrase:
        n_with_synset += 1

print(f'Phrases built: {len(word_phrase_lookup):,}')
print(f'  With WordNet definition: {n_with_synset:,} ({n_with_synset/len(word_phrase_lookup)*100:.1f}%)')
print(f'  Bare (no synset):        {len(word_phrase_lookup)-n_with_synset:,}')

# Show examples
print(f'\nExample phrases:')
for word in list(sorted(all_answer_words))[:5]:
    print(f'  {word}: "{word_phrase_lookup[word]}"')


## §3 — Assembling the Triplet Training Dataset

Each training triplet consists of three text components:

| Component | Description | CALE Input |
|-----------|-------------|------------|
| **Anchor** $c$ | Definition within clue context | `"Parties broken for <t>sea-faring group</t>"` |
| **Positive** $a$ | Correct answer (decontextualized) | `"<t>fleet</t>: a group of ships"` |
| **Negative** $f$ | Distractor (decontextualized) | `"<t>crew</t>: the people on a ship"` |

The anchor carries the cryptic misdirection signal; the positive and negative are
neutral, decontextualized representations. The triplet loss will push the anchor
closer to the positive and farther from the negative in embedding space.


In [ ]:
# ============================================================
# §3.1 — Match Real Pairs with Distractors to Form Triplets
# ============================================================
# In dataset_harder.parquet, each real pair (label=1) has a
# corresponding distractor (label=0) sharing the same clue_id.
#
# For label=0 rows:
#   - answer    = the original real answer (ALL CAPS, same as label=1)
#   - answer_wn = the DISTRACTOR word (lowercase)
#
# We join on clue_id + definition_wn to pair them.

df_dist_for_merge = df_dist[['clue_id', 'definition_wn', 'answer_wn']].rename(
    columns={'answer_wn': 'distractor_wn'}
)

df_train_triplets = df_train_real.merge(
    df_dist_for_merge,
    on=['clue_id', 'definition_wn'],
    how='inner'
)
print(f'Training triplets after real↔distractor merge: {len(df_train_triplets):,}')

# Assign CALE phrases for positive (real answer) and negative (distractor)
df_train_triplets['positive_phrase'] = df_train_triplets['answer_wn'].map(word_phrase_lookup)
df_train_triplets['negative_phrase'] = df_train_triplets['distractor_wn'].map(word_phrase_lookup)

# Drop rows where any phrase is missing
before = len(df_train_triplets)
df_train_triplets = df_train_triplets.dropna(
    subset=['cale_phrase', 'positive_phrase', 'negative_phrase']
).copy()
after = len(df_train_triplets)
print(f'After phrase filter: {after:,} (dropped {before - after:,})')

if SAMPLE_MODE:
    unique_train_pairs = df_train_triplets['pair_key'].unique()
    sample_pairs = np.random.RandomState(RANDOM_SEED).choice(
        unique_train_pairs,
        size=min(SAMPLE_SIZE, len(unique_train_pairs)),
        replace=False
    )
    df_train_triplets = df_train_triplets[
        df_train_triplets['pair_key'].isin(sample_pairs)
    ].copy()
    print(f'SAMPLE MODE: using {len(df_train_triplets):,} triplet rows '
          f'({len(sample_pairs):,} unique pairs)')

print(f'\nFinal triplet dataset: {len(df_train_triplets):,} rows')

# Show example triplets
print(f'\nExample triplets:')
for _, row in df_train_triplets.head(3).iterrows():
    print(f'  Anchor:   "{row["cale_phrase"]}"')
    print(f'  Positive: "{row["positive_phrase"]}"')
    print(f'  Negative: "{row["negative_phrase"]}"')
    print()


In [ ]:
# ============================================================
# §3.1 — PyTorch Dataset for Triplet Training
# ============================================================
# Wraps our triplet DataFrame into a PyTorch Dataset that the DataLoader
# can iterate over. Each __getitem__ returns the three raw text strings;
# tokenization happens in the training loop (allows dynamic batching).

class TripletDataset(Dataset):
    '''PyTorch Dataset for (anchor, positive, negative) text triplets.

    Each item returns:
        anchor_text:   str — clue with <t></t> delimiters around definition
        positive_text: str — CALE phrase for correct answer
        negative_text: str — CALE phrase for distractor

    Tokenization is deferred to the training loop to allow the tokenizer's
    padding/truncation to operate at the batch level.
    '''

    def __init__(self, dataframe):
        self.anchors   = dataframe['cale_phrase'].values
        self.positives = dataframe['positive_phrase'].values
        self.negatives = dataframe['negative_phrase'].values

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, idx):
        return {
            'anchor':   self.anchors[idx],
            'positive': self.positives[idx],
            'negative': self.negatives[idx],
        }


triplet_dataset = TripletDataset(df_train_triplets)
triplet_loader  = DataLoader(
    triplet_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,       # 0 for Colab compatibility
    drop_last=True,      # avoid partial batches that destabilize BN
)

print(f'Triplet dataset: {len(triplet_dataset):,} samples')
print(f'Batches per epoch: {len(triplet_loader):,}')
print(f'Batch size: {TRAIN_BATCH_SIZE}')


---

# Phase 2: Fine-Tuning the CALE Model

## §4 — Model Architecture and Concept-Aligned Extraction

We load the pretrained CALE model (`gabrielloiseau/CALE-MBERT-en`) and "thaw" its
parameters for fine-tuning with triplet margin loss. The key architectural detail is
**concept-aligned extraction**: rather than using the [CLS] token or mean pooling over
all tokens, we extract the embedding for *only the tokens corresponding to the
definition span* (the text between `<t>` and `</t>` delimiters).

This is critical because:

1. CALE was specifically trained to produce sense-disambiguated embeddings for delimited
   target words — this is its entire design purpose (Loiseau et al., 2024).
2. We want the anchor embedding to represent *how the definition's meaning shifts when
   surrounded by the clue context* — not the meaning of the entire clue surface.
3. The positive and negative embeddings use [CLS] (or equivalently, the delimited word
   in a decontextualized phrase) — creating the asymmetry that the triplet loss exploits.

**From KCT's boilerplate:**

```python
def get_concept_aligned_embedding(model, encoded_input, def_start_char, def_end_char):
    outputs = model(**encoded_input)
    last_hidden_state = outputs.last_hidden_state  # [Batch, Seq_Len, Hidden_Dim]
    token_indices = [i for i in range(len(encoded_input['input_ids'][0]))
                     if encoded_input.token_to_chars(0, i) and
                        encoded_input.token_to_chars(0, i).start >= def_start_char and
                        encoded_input.token_to_chars(0, i).end <= def_end_char]
    concept_vector = last_hidden_state[:, token_indices, :].mean(dim=1)
    return concept_vector
```

We adapt this for batch processing and add the `<t></t>` delimiter detection to
automatically locate the definition span.


In [ ]:
# ============================================================
# §4.1 — Load Pretrained CALE Model and Tokenizer
# ============================================================

CALE_MODEL_NAME = 'gabrielloiseau/CALE-MBERT-en'

# Load the base transformer model (ModernBERT) and tokenizer.
# CALE is a fine-tuned ModernBERT model that produces sense-disambiguated
# embeddings for words wrapped in <t></t> delimiters.
tokenizer = AutoTokenizer.from_pretrained(CALE_MODEL_NAME)
model_cale = AutoModel.from_pretrained(CALE_MODEL_NAME)

# Move to GPU and set to training mode
model_cale.gradient_checkpointing_enable()
model_cale = model_cale.to(DEVICE)
model_cale.train()

# Verify the model loads correctly
n_params = sum(p.numel() for p in model_cale.parameters())
n_trainable = sum(p.numel() for p in model_cale.parameters() if p.requires_grad)
print(f'Model: {CALE_MODEL_NAME}')
print(f'Parameters: {n_params:,} total, {n_trainable:,} trainable')
print(f'Hidden dim: {model_cale.config.hidden_size}')
print(f'Device: {DEVICE}')


In [ ]:
# ============================================================
# §4.2 — Concept-Aligned Embedding Extraction (Batch Version)
# ============================================================
# Adapted from KCT's boilerplate to handle batched inputs.
#
# For the ANCHOR (clue text with <t></t> delimiters):
#   Extract the average of hidden states for tokens within the <t>...</t> span.
#   This gives us the definition's meaning AS MODIFIED BY the clue context.
#
# For POSITIVE and NEGATIVE (decontextualized CALE phrases):
#   Extract the average of hidden states for tokens within the <t>...</t> span.
#   Same mechanism, but the context is a neutral synset sentence, so the
#   embedding reflects the word's dictionary meaning.
#
# Note: We use <t></t> span extraction for ALL three components (not [CLS]),
# because CALE's training objective specifically optimizes the delimited-span
# representation. [CLS] in CALE doesn't carry the same semantic precision.

def find_delimiter_char_offsets(text):
    '''Find character offsets of the content between <t> and </t> delimiters.

    Returns (start, end) character indices of the CONTENT (excluding tags).
    These are offsets into the ORIGINAL text (before tokenization strips tags).

    Example:
        text = "Parties broken for <t>sea-faring group</t>"
        returns (19, 36)  # "sea-faring group" starts at 19
    '''
    start_tag = '<t>'
    end_tag = '</t>'

    start_pos = text.find(start_tag)
    if start_pos == -1:
        return None, None

    content_start = start_pos + len(start_tag)
    end_pos = text.find(end_tag, content_start)
    if end_pos == -1:
        return None, None

    return content_start, end_pos


def extract_concept_embedding(model, tokenizer, texts, device):
    '''Extract concept-aligned embeddings for a batch of CALE-delimited texts.

    For each text, identifies the <t></t> span, maps it to token indices,
    and averages the hidden states of those tokens.

    Args:
        model: The CALE transformer model
        tokenizer: The CALE tokenizer
        texts: list of str — texts with <t></t> delimiters
        device: torch device

    Returns:
        torch.Tensor of shape (batch_size, hidden_dim)
    '''
    # Tokenize the full batch
    encoded = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128,
    ).to(device)

    # Forward pass
    outputs = model(**encoded)
    hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden_dim)

    batch_size = hidden_states.shape[0]
    hidden_dim = hidden_states.shape[2]
    concept_vectors = torch.zeros(batch_size, hidden_dim, device=device)

    for i in range(batch_size):
        text = texts[i]
        start_char, end_char = find_delimiter_char_offsets(text)

        if start_char is None:
            # No delimiters found — fall back to mean pooling (shouldn't happen
            # if data is clean, but defensive coding)
            attention_mask = encoded['attention_mask'][i]
            concept_vectors[i] = (
                hidden_states[i] * attention_mask.unsqueeze(-1)
            ).sum(dim=0) / attention_mask.sum()
            continue

        # Map character offsets to token indices.
        # A token belongs to the definition span if its character span
        # falls within [start_char, end_char).
        token_indices = []
        for tok_idx in range(encoded['input_ids'].shape[1]):
            span = encoded.token_to_chars(i, tok_idx)
            if span is None:
                continue  # special tokens ([CLS], [SEP], [PAD])
            if span.start >= start_char and span.end <= end_char:
                token_indices.append(tok_idx)

        if token_indices:
            concept_vectors[i] = hidden_states[i, token_indices, :].mean(dim=0)
        else:
            # Fallback: use all non-padding tokens
            attention_mask = encoded['attention_mask'][i]
            concept_vectors[i] = (
                hidden_states[i] * attention_mask.unsqueeze(-1)
            ).sum(dim=0) / attention_mask.sum()

    return concept_vectors


# Quick sanity check with a known example
test_text = "Parties broken for <t>sea-faring group</t>"
start, end = find_delimiter_char_offsets(test_text)
print(f'Test: "{test_text}"')
print(f'  Delimiter content: "{test_text[start:end]}" (chars {start}-{end})')


## §5 — Training Loop

The training objective is **triplet margin loss** (Schroff et al., 2015):

$$\mathcal{L} = \max(0,\ \|z_c - z_a\|_2 - \|z_c - z_f\|_2 + \alpha)$$

where:
- $z_c$ = concept-aligned anchor (contextualized definition)
- $z_a$ = positive (correct answer, decontextualized)
- $z_f$ = negative (distractor, decontextualized)
- $\alpha$ = margin (we use 1.0, following KCT's boilerplate)

The loss pushes the anchor **closer to the correct answer** and **further from the
distractor** by at least the margin $\alpha$. Over many iterations, this teaches the model
to "see through" the misleading clue context and map the definition toward its true
semantic target.

**Training considerations:**
- **Learning rate**: 2e-5 (standard for transformer fine-tuning; aggressive rates
  destabilize pretrained weights)
- **Optimizer**: AdamW (weight decay prevents the model from drifting too far from CALE's
  pretrained distribution)
- **Epochs**: 3–5 (transformer fine-tuning typically converges quickly; more risks
  overfitting to training distractors)
- **Gradient accumulation**: Optional for larger effective batch sizes on limited VRAM
- **Early stopping**: Monitor validation loss on a held-aside portion of training data
  (NOT the test set)


In [ ]:
# ============================================================
# §5.1 — Training Configuration
# ============================================================

# Hyperparameters — following KCT's recommendations and standard
# transformer fine-tuning practice.
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
MARGIN        = 1.0       # Triplet margin (KCT boilerplate default)
NUM_EPOCHS    = 3         # Start conservative; increase if loss plateaus
GRAD_ACCUM    = 1         # Gradient accumulation steps (increase for small batch)

# Loss function
triplet_loss_fn = nn.TripletMarginLoss(margin=MARGIN, p=2)

# Optimizer — AdamW with weight decay to regularize fine-tuning
optimizer = torch.optim.AdamW(
    model_cale.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler — linear warmup then decay
# Warmup helps stabilize early training when gradients are large
total_steps = len(triplet_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    return max(0.0, 1.0 - (step - warmup_steps) / (total_steps - warmup_steps))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f'Training configuration:')
print(f'  Learning rate:  {LEARNING_RATE}')
print(f'  Weight decay:   {WEIGHT_DECAY}')
print(f'  Margin:         {MARGIN}')
print(f'  Epochs:         {NUM_EPOCHS}')
print(f'  Total steps:    {total_steps:,}')
print(f'  Warmup steps:   {warmup_steps:,}')


In [ ]:
%%time
# ============================================================
# §5.2B — Training Loop (Refined for Memory Stability)
# ============================================================
import os
import torch
from torch.amp import autocast, GradScaler

# Memory management
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
scaler = GradScaler()
training_log = []

for epoch in range(NUM_EPOCHS):
    model_cale.train()
    epoch_losses = []
    t0_epoch = time.time()

    # Reset optimizer
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(triplet_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')):

        # --- Forward Pass with Mixed Precision ---
        with autocast(device_type='cuda', dtype=torch.float16):
            z_anchor = extract_concept_embedding(model_cale, tokenizer, batch['anchor'], DEVICE)
            z_positive = extract_concept_embedding(model_cale, tokenizer, batch['positive'], DEVICE)
            z_negative = extract_concept_embedding(model_cale, tokenizer, batch['negative'], DEVICE)

            loss = triplet_loss_fn(z_anchor, z_positive, z_negative)

            if GRAD_ACCUM > 1:
                loss = loss / GRAD_ACCUM

        # --- Backward Pass ---
        scaler.scale(loss).backward()

        # --- Optimizer Step ---
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_cale.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            # Use set_to_none=True to free gradient memory
            optimizer.zero_grad(set_to_none=True)

        # Log the loss: detach and cast to float to prevent memory leak and UnpicklingError
        detached_loss = float(loss.detach().item())
        epoch_losses.append(detached_loss * GRAD_ACCUM)

        # --- CRITICAL: Manual memory cleanup for small GPUs ---
        del z_anchor, z_positive, z_negative, loss

        if (step + 1) % 100 == 0:
            avg_loss = np.mean(epoch_losses[-100:])
            lr_current = scheduler.get_last_lr()[0]
            training_log.append({
                'epoch': epoch + 1,
                'step': step + 1,
                'loss': avg_loss,
                'lr': lr_current,
            })
            print(f'  Step {step+1:>5d} | Loss: {avg_loss:.4f} | LR: {lr_current:.2e}')

    # --- End of Epoch ---
    epoch_loss = float(np.mean(epoch_losses))
    print(f'\nEpoch {epoch+1} complete: avg loss = {epoch_loss:.4f}, time = {time.time() - t0_epoch:.0f}s')

    # Save Checkpoint (with safe Python types)
    torch.save({
        'epoch': int(epoch + 1),
        'model_state_dict': model_cale.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss,
    }, MODELS_DIR / f'cale_finetuned_epoch{epoch+1}.pt')

In [ ]:
# Old training loop, poor memory management
'''
%%time
# ============================================================
# §5.2 — Training Loop
# ============================================================
import os
from torch.amp import autocast
from torch.amp import autocast, GradScaler

scaler = GradScaler()

# Fix 4: reduce memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

training_log = []
for epoch in range(NUM_EPOCHS):
    model_cale.train()
    epoch_losses = []
    t0_epoch = time.time()
    for step, batch in enumerate(tqdm(triplet_loader,
                                       desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')):
        # --- Extract embeddings (Fix 1: mixed precision, Fix 3: cache clearing) ---
        with autocast(device_type='cuda', dtype=torch.float16):
            z_anchor = extract_concept_embedding(
                model_cale, tokenizer, batch['anchor'], DEVICE
            )
        torch.cuda.empty_cache()  # Fix 3

        with autocast(device_type='cuda', dtype=torch.float16):
            z_positive = extract_concept_embedding(
                model_cale, tokenizer, batch['positive'], DEVICE
            )
        torch.cuda.empty_cache()  # Fix 3

        with autocast(device_type='cuda', dtype=torch.float16):
            z_negative = extract_concept_embedding(
                model_cale, tokenizer, batch['negative'], DEVICE
            )

        # --- Compute triplet loss ---
        with autocast(device_type='cuda', dtype=torch.float16):
            z_anchor = extract_concept_embedding(
                model_cale, tokenizer, batch['anchor'], DEVICE
            )
        torch.cuda.empty_cache()

        with autocast(device_type='cuda', dtype=torch.float16):
            z_positive = extract_concept_embedding(
                model_cale, tokenizer, batch['positive'], DEVICE
            )
        torch.cuda.empty_cache()

        with autocast(device_type='cuda', dtype=torch.float16):
            z_negative = extract_concept_embedding(
                model_cale, tokenizer, batch['negative'], DEVICE
            )

        with autocast(device_type='cuda', dtype=torch.float16):
            loss = triplet_loss_fn(z_anchor, z_positive, z_negative)

        # --- Backpropagation ---
        if GRAD_ACCUM > 1:
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_cale.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            torch.cuda.empty_cache()

        epoch_losses.append(loss.item() * (GRAD_ACCUM if GRAD_ACCUM > 1 else 1))
        if (step + 1) % 100 == 0:
            avg_loss = np.mean(epoch_losses[-100:])
            lr_current = scheduler.get_last_lr()[0]
            training_log.append({
                'epoch': epoch + 1,
                'step': step + 1,
                'loss': avg_loss,
                'lr': lr_current,
            })
            print(f'  Step {step+1:>5d} | Loss: {avg_loss:.4f} | LR: {lr_current:.2e}')

    epoch_loss = np.mean(epoch_losses)
    epoch_time = time.time() - t0_epoch
    print(f'\nEpoch {epoch+1} complete: avg loss = {epoch_loss:.4f}, '
          f'time = {epoch_time:.0f}s')
    ckpt_path = MODELS_DIR / f'cale_finetuned_epoch{epoch+1}.pt'
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model_cale.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': epoch_loss,
    }, ckpt_path)
    print(f'  Checkpoint saved: {ckpt_path}')

print(f'\nTraining complete!')
'''

In [ ]:
# ============================================================
# §5.3 — Training Loss Curve
# ============================================================

if training_log:
    df_log = pd.DataFrame(training_log)

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(range(len(df_log)), df_log['loss'], 'b-', alpha=0.7)
    ax.set_xlabel('Logging Step (every 100 batches)')
    ax.set_ylabel('Triplet Margin Loss')
    ax.set_title('CALE Fine-Tuning: Training Loss')
    ax.grid(True, alpha=0.3)

    # Mark epoch boundaries
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_steps = df_log[df_log['epoch'] == epoch]
        if not epoch_steps.empty:
            ax.axvline(x=epoch_steps.index[-1], color='r', linestyle='--',
                       alpha=0.5, label=f'Epoch {epoch} end' if epoch == 1 else '')

    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Loss curve saved to {OUTPUT_DIR / "training_loss_curve.png"}')
else:
    print('No training log entries — training may not have run yet.')


In [ ]:
# ============================================================
# §6.1 — Load the Best Checkpoint
# ============================================================
# Select the final epoch checkpoint (or the one with lowest validation loss
# if early stopping was used).

# Add both required NumPy types to the allowlist
torch.serialization.add_safe_globals([
    np._core.multiarray.scalar,
    np.dtype
])

best_ckpt = MODELS_DIR / f'cale_finetuned_epoch{NUM_EPOCHS}.pt'
if best_ckpt.exists():
    checkpoint = torch.load(best_ckpt, map_location=DEVICE)
    model_cale.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded checkpoint: {best_ckpt} (epoch {checkpoint["epoch"]}, '
          f'loss {checkpoint["loss"]:.4f})')
else:
    print(f'WARNING: Checkpoint not found at {best_ckpt}')
    print(f'  Using current model state (may be untrained if training was skipped)')

model_cale.eval()

# Also load a FRESH stock CALE for comparison
model_stock = AutoModel.from_pretrained(CALE_MODEL_NAME)
model_stock = model_stock.to(DEVICE)
model_stock.eval()
print(f'Stock CALE loaded for comparison')

---

# Phase 3: Comparing Learned $g$ vs. Stock CALE

## §6 — Re-embed with Learned Model

Now that we have a fine-tuned CALE model (learned $g$), we re-embed our data using both
the stock CALE and the learned $g$ to compare how the embedding space has changed.

**What to look for** (from KCT's meeting notes):
- *"Words more generic in CALE space may cluster more in learned embedding space"*
- *"Take a point in space and look in the neighborhood — neighborhood structure"*
- Do cryptic-crossword-specific word associations emerge?
- Does the learned model better separate definitions from their distractors?

We compute embeddings for the **test set only** (set $\mathbf{I}$) — these are the
embeddings we'll use for the ATE experiment in Phase 4.


In [ ]:
# ============================================================
# §6.2 — Generate Test-Set Embeddings with Both Models
# ============================================================
# For each test-set row, generate:
# - Clue-context embedding (definition within clue) — this is T=1
# - Decontextualized definition embedding (allsense) — this is T=0
# - Answer embedding (allsense) — for retrieval ranking
#
# We do this with BOTH models so we can compare the misdirection ATE.
# Phrases are built on-the-fly from the word_phrase_lookup created in §2.2.

def batch_embed(model, tokenizer, texts, device, batch_size=64):
    '''Generate concept-aligned embeddings for a list of texts.

    Processes in batches to avoid OOM on large datasets.
    Returns numpy array of shape (N, hidden_dim).
    '''
    model.eval()
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size),
                      desc='Embedding', leave=False):
        batch_texts = texts[start:start+batch_size]

        with torch.no_grad():
            emb = extract_concept_embedding(model, tokenizer, batch_texts, device)

        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)


# Prepare test-set texts — all constructed from parquet columns + WordNet
test_clue_phrases = df_test_real['cale_phrase'].tolist()
test_def_phrases  = [word_phrase_lookup.get(d, f'<t>{d}</t>')
                     for d in df_test_real['definition_wn']]
test_ans_phrases  = [word_phrase_lookup.get(a, f'<t>{a}</t>')
                     for a in df_test_real['answer_wn']]

if SAMPLE_MODE:
    sample_n = min(SAMPLE_SIZE, len(test_clue_phrases))
    sample_idx = np.random.RandomState(RANDOM_SEED).choice(
        len(test_clue_phrases), sample_n, replace=False)
    test_clue_phrases_s = [test_clue_phrases[i] for i in sample_idx]
    test_def_phrases_s  = [test_def_phrases[i] for i in sample_idx]
    test_ans_phrases_s  = [test_ans_phrases[i] for i in sample_idx]
    df_test_s = df_test_real.iloc[sample_idx].copy()
    print(f'SAMPLE MODE: embedding {sample_n:,} test rows')
else:
    test_clue_phrases_s = test_clue_phrases
    test_def_phrases_s  = test_def_phrases
    test_ans_phrases_s  = test_ans_phrases
    df_test_s = df_test_real.copy()

print(f'\nEmbedding test set with LEARNED model...')
learned_clue_emb = batch_embed(model_cale, tokenizer, test_clue_phrases_s,
                                DEVICE, EMBED_BATCH_SIZE)
learned_def_emb  = batch_embed(model_cale, tokenizer, test_def_phrases_s,
                                DEVICE, EMBED_BATCH_SIZE)
learned_ans_emb  = batch_embed(model_cale, tokenizer, test_ans_phrases_s,
                                DEVICE, EMBED_BATCH_SIZE)

print(f'Embedding test set with STOCK model...')
stock_clue_emb = batch_embed(model_stock, tokenizer, test_clue_phrases_s,
                              DEVICE, EMBED_BATCH_SIZE)
stock_def_emb  = batch_embed(model_stock, tokenizer, test_def_phrases_s,
                              DEVICE, EMBED_BATCH_SIZE)
stock_ans_emb  = batch_embed(model_stock, tokenizer, test_ans_phrases_s,
                              DEVICE, EMBED_BATCH_SIZE)

print(f'\nEmbedding shapes:')
print(f'  Learned clue-context: {learned_clue_emb.shape}')
print(f'  Learned decontext:    {learned_def_emb.shape}')
print(f'  Learned answer:       {learned_ans_emb.shape}')
print(f'  Stock clue-context:   {stock_clue_emb.shape}')
print(f'  Stock decontext:      {stock_def_emb.shape}')
print(f'  Stock answer:         {stock_ans_emb.shape}')


## §7 — Embedding Space Analysis

We compare the two embedding spaces (stock vs. learned) across several dimensions to
understand what the fine-tuning has changed. This addresses KCT's question: *"What has
evolved during training?"*


In [ ]:
# ============================================================
# §7.1 — Cosine Similarity Distributions: Context vs. Decontext
# ============================================================
# The core misdirection signal: how much does clue context shift the
# definition embedding AWAY from the answer?
#
# For each (definition, answer) pair, compute:
#   sim_decontext = cos(def_allsense, ans_allsense)  — baseline (T=0)
#   sim_context   = cos(def_clue_context, ans_allsense) — treatment (T=1)
#   delta = sim_context - sim_decontext
#
# Negative delta = misdirection (context hurts).
# We expect the LEARNED model to show LESS negative delta (it has learned
# to partially counteract the misdirection).

def rowwise_cosine(A, B):
    '''Compute per-row cosine similarity between two matrices.'''
    # Normalize rows
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return np.sum(A_norm * B_norm, axis=1)

# Stock CALE similarities
stock_sim_decontext = rowwise_cosine(stock_def_emb, stock_ans_emb)
stock_sim_context   = rowwise_cosine(stock_clue_emb, stock_ans_emb)
stock_delta         = stock_sim_context - stock_sim_decontext

# Learned model similarities
learned_sim_decontext = rowwise_cosine(learned_def_emb, learned_ans_emb)
learned_sim_context   = rowwise_cosine(learned_clue_emb, learned_ans_emb)
learned_delta         = learned_sim_context - learned_sim_decontext

print('Cosine Similarity Summary (definition → answer):')
print(f'{"":30s} {"Stock CALE":>12s}  {"Learned g":>12s}')
print(f'{"-"*56}')
print(f'{"Decontextualized (T=0) mean":30s} {stock_sim_decontext.mean():>12.4f}  '
      f'{learned_sim_decontext.mean():>12.4f}')
print(f'{"Contextualized (T=1) mean":30s} {stock_sim_context.mean():>12.4f}  '
      f'{learned_sim_context.mean():>12.4f}')
print(f'{"Delta (T=1 minus T=0) mean":30s} {stock_delta.mean():>12.4f}  '
      f'{learned_delta.mean():>12.4f}')
print(f'{"Delta median":30s} {np.median(stock_delta):>12.4f}  '
      f'{np.median(learned_delta):>12.4f}')
print(f'{"% pairs with negative delta":30s} '
      f'{(stock_delta < 0).mean()*100:>11.1f}%  '
      f'{(learned_delta < 0).mean()*100:>11.1f}%')


In [ ]:
# ============================================================
# §7.2 — Visualization: Delta Distributions
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Stock CALE delta distribution
axes[0].hist(stock_delta, bins=80, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=1)
axes[0].axvline(x=np.median(stock_delta), color='red', linestyle='--',
                label=f'Median = {np.median(stock_delta):.4f}')
axes[0].set_xlabel('Δ Cosine Similarity (context − decontext)')
axes[0].set_ylabel('Count')
axes[0].set_title('Stock CALE: Misdirection Delta')
axes[0].legend()

# Learned model delta distribution
axes[1].hist(learned_delta, bins=80, alpha=0.7, color='darkorange', edgecolor='white')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=1)
axes[1].axvline(x=np.median(learned_delta), color='red', linestyle='--',
                label=f'Median = {np.median(learned_delta):.4f}')
axes[1].set_xlabel('Δ Cosine Similarity (context − decontext)')
axes[1].set_title('Learned g: Misdirection Delta')
axes[1].legend()

plt.suptitle('Misdirection Effect: How Much Does Clue Context Shift Definition→Answer Similarity?',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'delta_distributions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# §7.3 — Neighborhood Structure Analysis
# ============================================================
# KCT suggested examining how neighborhood structure changes:
# "Take a point in space and look in the neighborhood."
#
# For a sample of definitions, find the k-nearest answers in both
# embedding spaces and see how the neighborhoods differ.

from sklearn.neighbors import NearestNeighbors

# Build answer embedding indices for both models
# Using a sample of unique answers for efficiency
unique_test_answers = df_test_s['answer_wn'].unique()

# For a sample of definitions, find nearest answers in both spaces
sample_defs = df_test_s[['definition_wn']].drop_duplicates().sample(
    n=min(200, df_test_s['definition_wn'].nunique()),
    random_state=RANDOM_SEED
)

K_NEIGHBORS = 10

# This is a simplified version — in practice you'd build a full
# answer-embedding index. Here we compare neighborhoods for the
# test-set rows directly.
print(f'Neighborhood analysis on {len(sample_defs)} definitions, k={K_NEIGHBORS}')
print(f'\nComparing how many of the top-{K_NEIGHBORS} nearest answers')
print(f'overlap between stock CALE and learned g...')

# Compute full pairwise for the sample
# (In production, use FAISS or annoy for efficiency)
overlap_scores = []

for _, def_row in sample_defs.iterrows():
    d_wn = def_row['definition_wn']
    mask = df_test_s['definition_wn'] == d_wn
    if mask.sum() == 0:
        continue

    # Get this definition's decontextualized embedding from both models
    idx = mask.values.nonzero()[0][0]

    stock_d_vec = stock_def_emb[idx:idx+1]
    learned_d_vec = learned_def_emb[idx:idx+1]

    # Similarities to all test answers
    stock_sims = cosine_similarity(stock_d_vec, stock_ans_emb)[0]
    learned_sims = cosine_similarity(learned_d_vec, learned_ans_emb)[0]

    stock_topk = set(np.argsort(stock_sims)[::-1][:K_NEIGHBORS])
    learned_topk = set(np.argsort(learned_sims)[::-1][:K_NEIGHBORS])

    overlap = len(stock_topk & learned_topk) / K_NEIGHBORS
    overlap_scores.append(overlap)

if overlap_scores:
    print(f'\nNeighborhood overlap (top-{K_NEIGHBORS}):')
    print(f'  Mean overlap:   {np.mean(overlap_scores):.3f}')
    print(f'  Median overlap: {np.median(overlap_scores):.3f}')
    print(f'  Min overlap:    {np.min(overlap_scores):.3f}')
    print(f'  Max overlap:    {np.max(overlap_scores):.3f}')
    print(f'\nInterpretation: Lower overlap = learned g reorganized neighborhoods more.')
    print(f'Values near 1.0 = fine-tuning preserved neighborhood structure.')
    print(f'Values near 0.0 = dramatically different neighborhoods.')


---

# Phase 4: Average Treatment Effect Estimation

## §8 — Experimental Design

This is the culmination of the notebook: a clean, one-shot estimation of the misdirection
Average Treatment Effect using both stock CALE and learned $g$ on **held-out test data**.

Following Egami et al.'s procedure:

1. $g$ (the codebook function) was discovered and trained on training data $\mathbf{J}$
2. $g$ is now **locked** — no further modifications
3. We apply $g$ to test data $\mathbf{I}$ to produce embeddings
4. We estimate the ATE: $\widehat{\text{ATE}} = \bar{Y}(T=1) - \bar{Y}(T=0)$

**Our estimand:**

For each (definition, answer) pair in the test set, define:
- $Y_i(T=1)$: cosine similarity between definition-in-clue-context and answer
- $Y_i(T=0)$: cosine similarity between decontextualized definition and answer

Then: $\text{ATE} = \mathbb{E}[Y_i(T=1) - Y_i(T=0)]$

A **negative ATE** indicates misdirection: clue context degrades the model's ability to
link the definition to the correct answer.

**Hypothesis:** The learned $g$ should show a *less negative* (closer to zero) ATE than
stock CALE, because fine-tuning has taught the model to partially counteract misdirection.

We also run a **retrieval-based** ATE for comparison with M2's primary misdirection
evidence: for each definition, rank all candidate answers by cosine similarity and record
the rank of the true answer under both treatment conditions.


In [ ]:
# ============================================================
# §8.1 — Cosine-Based ATE Estimation
# ============================================================
# This is the simplest and most direct ATE measure.
# We already computed all the required embeddings in §6.2.

print('='*65)
print('AVERAGE TREATMENT EFFECT: Cosine Similarity')
print('='*65)

# Stock CALE ATE
stock_ate = stock_delta.mean()
stock_ate_se = stock_delta.std() / np.sqrt(len(stock_delta))
stock_ate_ci = (stock_ate - 1.96 * stock_ate_se,
                stock_ate + 1.96 * stock_ate_se)

# Learned g ATE
learned_ate = learned_delta.mean()
learned_ate_se = learned_delta.std() / np.sqrt(len(learned_delta))
learned_ate_ci = (learned_ate - 1.96 * learned_ate_se,
                  learned_ate + 1.96 * learned_ate_se)

print(f'\n{"Model":20s} {"ATE":>10s} {"SE":>10s} {"95% CI":>24s}')
print(f'{"-"*66}')
print(f'{"Stock CALE":20s} {stock_ate:>10.4f} {stock_ate_se:>10.4f} '
      f'[{stock_ate_ci[0]:>10.4f}, {stock_ate_ci[1]:>10.4f}]')
print(f'{"Learned g":20s} {learned_ate:>10.4f} {learned_ate_se:>10.4f} '
      f'[{learned_ate_ci[0]:>10.4f}, {learned_ate_ci[1]:>10.4f}]')
print(f'{"Difference":20s} {learned_ate - stock_ate:>10.4f}')

print(f'\nInterpretation:')
print(f'  Negative ATE = misdirection (context hurts prediction)')
print(f'  Less negative = model partially counteracts misdirection')
if learned_ate > stock_ate:
    print(f'  → Learned g shows LESS misdirection than stock CALE '
          f'(delta = {learned_ate - stock_ate:+.4f})')
else:
    print(f'  → Learned g shows MORE misdirection than stock CALE '
          f'(delta = {learned_ate - stock_ate:+.4f})')
    print(f'    This could indicate overfitting or that fine-tuning')
    print(f'    amplified contextual sensitivity without correcting direction.')


In [ ]:
# ============================================================
# §8.2 — Retrieval-Based ATE Estimation
# ============================================================
# Following M2's retrieval analysis (NB 04): for each definition, rank
# ALL candidate answers by cosine similarity and record the rank of the
# true answer. Compare ranks under T=0 (decontextualized) and T=1 (context).
#
# This is more robust than pairwise cosine because it measures how the
# definition moves relative to the ENTIRE answer space, not just one pair.

def compute_retrieval_ranks(def_embeddings, ans_embeddings, true_answer_indices,
                            batch_size=500):
    '''Compute the rank of the true answer for each definition query.

    For each definition embedding, rank all answer embeddings by cosine
    similarity and return the 1-indexed rank of the true answer.
    '''
    n_queries = len(def_embeddings)
    ranks = np.zeros(n_queries, dtype=int)

    for start in tqdm(range(0, n_queries, batch_size),
                      desc='Retrieval ranks', leave=False):
        end = min(start + batch_size, n_queries)
        batch_def = def_embeddings[start:end]

        # Cosine similarity to all answers
        sims = cosine_similarity(batch_def, ans_embeddings)  # (batch, n_answers)

        for i, row_sims in enumerate(sims):
            true_idx = true_answer_indices[start + i]
            # Rank = number of answers with higher similarity + 1
            ranks[start + i] = (row_sims > row_sims[true_idx]).sum() + 1

    return ranks


# Build answer index for test set
# Use ALL unique answers as the candidate pool (same as M2's 45,254)
n_test = len(df_test_s)

# Map each test row's answer to its index in the answer embedding array
test_true_ans_indices = np.arange(n_test)  # Each row's own answer embedding

print('Computing retrieval ranks...\n')

# Stock CALE: decontextualized (T=0)
print('Stock CALE, T=0 (decontextualized):')
stock_ranks_t0 = compute_retrieval_ranks(
    stock_def_emb, stock_ans_emb, test_true_ans_indices)

# Stock CALE: contextualized (T=1)
print('Stock CALE, T=1 (clue context):')
stock_ranks_t1 = compute_retrieval_ranks(
    stock_clue_emb, stock_ans_emb, test_true_ans_indices)

# Learned g: decontextualized (T=0)
print('Learned g, T=0 (decontextualized):')
learned_ranks_t0 = compute_retrieval_ranks(
    learned_def_emb, learned_ans_emb, test_true_ans_indices)

# Learned g: contextualized (T=1)
print('Learned g, T=1 (clue context):')
learned_ranks_t1 = compute_retrieval_ranks(
    learned_clue_emb, learned_ans_emb, test_true_ans_indices)

# Retrieval ATE: change in median rank
print(f'\n{"="*65}')
print(f'AVERAGE TREATMENT EFFECT: Retrieval Rank')
print(f'{"="*65}')
print(f'\n{"Model":20s} {"Median T=0":>12s} {"Median T=1":>12s} '
      f'{"Delta":>10s} {"% Worsened":>12s}')
print(f'{"-"*68}')

stock_rank_delta = stock_ranks_t1.astype(float) - stock_ranks_t0.astype(float)
learned_rank_delta = learned_ranks_t1.astype(float) - learned_ranks_t0.astype(float)

print(f'{"Stock CALE":20s} '
      f'{np.median(stock_ranks_t0):>12.0f} '
      f'{np.median(stock_ranks_t1):>12.0f} '
      f'{np.median(stock_rank_delta):>+10.0f} '
      f'{(stock_rank_delta > 0).mean()*100:>11.1f}%')
print(f'{"Learned g":20s} '
      f'{np.median(learned_ranks_t0):>12.0f} '
      f'{np.median(learned_ranks_t1):>12.0f} '
      f'{np.median(learned_rank_delta):>+10.0f} '
      f'{(learned_rank_delta > 0).mean()*100:>11.1f}%')

print(f'\nPositive delta = context worsened rank (misdirection)')
print(f'Smaller delta for learned g = fine-tuning reduced misdirection')


In [ ]:
# ============================================================
# §8.3 — ATE Visualization
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Cosine ATE comparison
models = ['Stock CALE', 'Learned g']
ates = [stock_ate, learned_ate]
cis = [stock_ate_ci, learned_ate_ci]
colors = ['steelblue', 'darkorange']

for i, (model, ate, ci, color) in enumerate(zip(models, ates, cis, colors)):
    axes[0].barh(i, ate, color=color, alpha=0.8, edgecolor='black')
    axes[0].errorbar(ate, i, xerr=[[ate - ci[0]], [ci[1] - ate]],
                     fmt='none', color='black', capsize=5)
axes[0].axvline(x=0, color='black', linewidth=1)
axes[0].set_yticks(range(len(models)))
axes[0].set_yticklabels(models)
axes[0].set_xlabel('ATE (cosine similarity delta)')
axes[0].set_title('Cosine Similarity ATE\n(Negative = Misdirection)')

# Panel 2: Retrieval rank delta comparison
rank_data = [stock_rank_delta, learned_rank_delta]
bp = axes[1].boxplot(rank_data, labels=models, patch_artist=True,
                     showfliers=False, widths=0.5)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(y=0, color='black', linewidth=1)
axes[1].set_ylabel('Rank Delta (T=1 minus T=0)')
axes[1].set_title('Retrieval Rank ATE\n(Positive = Misdirection)')

plt.suptitle('Misdirection Average Treatment Effect: Stock CALE vs. Learned g',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---

## §9 — Summary and Next Steps

### Results Summary

This notebook implemented the four-phase pipeline for learning a custom codebook function
$g$ and comparing its misdirection properties against stock CALE, using `dataset_harder.parquet`
as the primary input.

**Key advantages of starting from dataset_harder.parquet:**
- Distractors are pre-constructed using cosine-similarity-based top-100 matching (Decision 6),
  ensuring consistency with M2's binary classification experiments.
- The harder dataset's distractors are semantically challenging — exactly what triplet training
  needs for effective contrastive learning.
- Eliminates redundant distractor generation, reducing notebook complexity and runtime.

**Phases completed:**
1. **Triplet construction:** Extracted (anchor, positive, negative) triplets by pairing
   real definition–answer pairs (label=1) with their matched distractors (label=0).
2. **Fine-tuning:** Trained CALE via triplet margin loss to learn a codebook function $g$
   that pulls contextualized definitions closer to correct answers.
3. **Embedding comparison:** Compared stock CALE vs. learned $g$ on cosine similarity
   distributions, delta distributions, and neighborhood structure.
4. **ATE estimation:** Estimated the misdirection Average Treatment Effect under both
   models using cosine-based and retrieval-based measures.

### Connections to M2 Results

The harder dataset experiments in NB 07 showed a +5.5 to +9.4pp context gap (context
features substantially help classification). This notebook's learned $g$ approach
complements those findings by directly optimizing the embedding space rather than
training a downstream classifier.

### Next Steps

- Compare learned $g$'s ATE reduction against the classifier-based context gap from NB 07
- Run with SAMPLE_MODE=False for full-data results
- Experiment with training hyperparameters (margin, learning rate, epochs)
- Investigate whether structural distractors (a subset of the harder distractors that
  share WordNet relationships) benefit more from learned $g$ than cosine-only distractors


In [ ]:
# ============================================================
# Save final results
# ============================================================

results = {
    'n_test_rows': len(df_test_s),
    'sample_mode': SAMPLE_MODE,
    'input_dataset': 'dataset_harder.parquet',
    'n_harder_rows': len(df_harder),
    'n_train_triplets': len(df_train_triplets),
    'stock_ate_cosine': float(stock_ate),
    'stock_ate_cosine_se': float(stock_ate_se),
    'learned_ate_cosine': float(learned_ate),
    'learned_ate_cosine_se': float(learned_ate_se),
    'stock_median_rank_t0': int(np.median(stock_ranks_t0)),
    'stock_median_rank_t1': int(np.median(stock_ranks_t1)),
    'learned_median_rank_t0': int(np.median(learned_ranks_t0)),
    'learned_median_rank_t1': int(np.median(learned_ranks_t1)),
    'training_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'margin': MARGIN,
}

results_path = OUTPUT_DIR / 'ate_results_learned_g.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to {results_path}')
print(json.dumps(results, indent=2))


In [ ]:
# Convert this notebook with output figures to html doc.
# First open in GL "Editor" and delete "widgets" JSON section.
# ! jupyter nbconvert --to html 09_learned_g_misdirection.ipynb